# 44 代替モデル比較 (JP 株式)

| 項目 | 内容 |
|------|------|
| 入力 | `data/equity_jp_ohlcv.parquet`, `data/macro.parquet` |
| 予測ターゲット | `fwd_ret_5d` (5 営業日先リターン) |
| 評価指標 | Rank IC (スピアマン相関) |
| 比較モデル | LightGBM, Ridge, Random Forest, XGBoost, Lasso, アンサンブル |
| 出力 | 比較チャート・スタッキング IC・推奨モデル |

## 分析の流れ
1. 共通特徴量エンジニアリング (42_ と同一)
2. 共通ウォークフォワード関数で全モデルを公平評価
3. モデル複雑度 vs IC のトレードオフ分析
4. スタッキングアンサンブルの効果検証
5. 本番推奨モデルの選定

---
## 0. 環境セットアップ

In [ ]:
import warnings
import time
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import yaml
from scipy.stats import spearmanr, ttest_1samp
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler

try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print('XGBoost が利用可能です')
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost が見つかりません。pip install xgboost でインストール可能です。')

try:
    import japanize_matplotlib
except ImportError:
    pass

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print('Setup OK')

---
## 1. 設定・データ読み込み

In [ ]:
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR    = Path(cfg['paths']['data'])
FIGURES_DIR = Path(cfg['paths']['figures'])
TABLES_DIR  = Path(cfg['paths']['tables'])
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = cfg.get('random_seed', 42)
np.random.seed(RANDOM_SEED)

TARGET   = cfg['equity']['target']    # fwd_ret_5d
N_SPLITS = cfg['model']['n_splits']   # 5

# ベースライン LightGBM パラメータ (42_ と同一)
LGB_PARAMS = cfg['model']['params']

print(f'Target  : {TARGET}')
print(f'Splits  : {N_SPLITS}')
print(f'Tables  : {TABLES_DIR.resolve()}')

In [ ]:
df_raw   = pd.read_parquet(DATA_DIR / 'equity_jp_ohlcv.parquet')
df_macro = pd.read_parquet(DATA_DIR / 'macro.parquet')

df_raw['date']   = pd.to_datetime(df_raw['date'])
df_macro['date'] = pd.to_datetime(df_macro['date'])

print(f'OHLCV  : {len(df_raw):,} 行  {df_raw["symbol"].nunique()} 銘柄')
print(f'Macro  : {len(df_macro):,} 行  {df_macro.shape[1]-1} 指標')
print(f'期間   : {df_raw["date"].min().date()} ~ {df_raw["date"].max().date()}')
df_raw.head(3)

---
## 2. 特徴量エンジニアリング (42_ と同一)

In [ ]:
# =====================================================
# 特徴量計算関数 (42_equity_lgbm_walkforward と同一)
# =====================================================

def add_returns(df, windows=(1, 5, 20)):
    for w in windows:
        df[f'ret_{w}d']    = df.groupby('symbol')['close'].pct_change(w)
        df[f'logret_{w}d'] = np.log(
            df.groupby('symbol')['close'].transform(lambda x: x / x.shift(w))
        )
    return df


def add_forward_returns(df, windows=(1, 5, 20)):
    for w in windows:
        df[f'fwd_ret_{w}d'] = df.groupby('symbol')['close'].transform(
            lambda x: x.pct_change(w).shift(-w)
        )
    return df


def add_sma(df, windows=(5, 10, 20, 60)):
    for w in windows:
        df[f'sma_{w}'] = df.groupby('symbol')['close'].transform(
            lambda x: x.rolling(w, min_periods=1).mean()
        )
        df[f'ema_{w}'] = df.groupby('symbol')['close'].transform(
            lambda x: x.ewm(span=w, adjust=False).mean()
        )
    for w in windows:
        df[f'price_sma{w}_ratio'] = df['close'] / df[f'sma_{w}']
    return df


def add_rsi(df, windows=(14, 28)):
    for w in windows:
        delta = df.groupby('symbol')['close'].transform(lambda x: x.diff())
        gain  = delta.where(delta > 0, 0.0)
        loss  = -delta.where(delta < 0, 0.0)
        ag    = gain.groupby(df['symbol']).transform(
            lambda x: x.ewm(com=w - 1, min_periods=w).mean()
        )
        al    = loss.groupby(df['symbol']).transform(
            lambda x: x.ewm(com=w - 1, min_periods=w).mean()
        )
        rs = ag / al.replace(0, np.nan)
        df[f'rsi_{w}'] = 100 - (100 / (1 + rs))
    return df


def add_macd(df, fast=12, slow=26, signal=9):
    def _calc(x):
        ef = x.ewm(span=fast, adjust=False).mean()
        es = x.ewm(span=slow, adjust=False).mean()
        m  = ef - es
        s  = m.ewm(span=signal, adjust=False).mean()
        return pd.DataFrame({'macd': m, 'macd_signal': s, 'macd_hist': m - s})
    result = df.groupby('symbol')['close'].apply(_calc).reset_index(level=0, drop=True)
    return pd.concat([df, result], axis=1)


def add_bollinger(df, window=20, k=2.0):
    df['bb_mid']   = df.groupby('symbol')['close'].transform(
        lambda x: x.rolling(window, min_periods=window // 2).mean()
    )
    df['bb_std']   = df.groupby('symbol')['close'].transform(
        lambda x: x.rolling(window, min_periods=window // 2).std()
    )
    df['bb_upper'] = df['bb_mid'] + k * df['bb_std']
    df['bb_lower'] = df['bb_mid'] - k * df['bb_std']
    band = (df['bb_upper'] - df['bb_lower']).replace(0, np.nan)
    df['bb_pct']   = (df['close'] - df['bb_lower']) / band
    df['bb_width'] = band / df['bb_mid']
    return df


def add_volume(df):
    df['vol_change']  = df.groupby('symbol')['volume'].transform(lambda x: x.pct_change())
    df['vol_sma20']   = df.groupby('symbol')['volume'].transform(
        lambda x: x.rolling(20, min_periods=5).mean()
    )
    df['vol_ratio']   = df['volume'] / df['vol_sma20'].replace(0, np.nan)
    df['turnover']    = df['close'] * df['volume']
    df['turnover_sma5'] = df.groupby('symbol')['turnover'].transform(
        lambda x: x.rolling(5, min_periods=2).mean()
    )
    return df


def add_volatility(df, windows=(5, 20)):
    lr = np.log(
        df.groupby('symbol')['close'].transform(lambda x: x / x.shift(1))
    )
    for w in windows:
        df[f'vol_{w}d'] = lr.groupby(df['symbol']).transform(
            lambda x: x.rolling(w, min_periods=w // 2).std() * np.sqrt(252)
        )
    return df


def add_cs_rank(df, cols):
    for col in cols:
        if col in df.columns:
            df[f'{col}_rank'] = df.groupby('date')[col].rank(pct=True)
    return df


def merge_macro(df, df_m):
    df = df.merge(df_m, on='date', how='left')
    macro_cols = [c for c in df_m.columns if c != 'date']
    df[macro_cols] = df[macro_cols].ffill()
    for col in macro_cols:
        df[f'{col}_ret1d'] = df[col].pct_change()
        df[f'{col}_ret5d'] = df[col].pct_change(5)
    return df


print('関数定義 OK')

In [ ]:
print('特徴量構築中...')
df = df_raw.copy().sort_values(['symbol', 'date']).reset_index(drop=True)
df = add_returns(df)
df = add_forward_returns(df)
df = add_sma(df)
df = add_rsi(df)
df = add_macd(df)
df = add_bollinger(df)
df = add_volume(df)
df = add_volatility(df)
df = add_cs_rank(df, ['ret_1d', 'ret_5d', 'ret_20d', 'vol_20d', 'rsi_14', 'vol_ratio'])
df = merge_macro(df, df_macro)

# 特徴量リスト
EXCLUDE = {'symbol', 'asset_class', 'date', 'open', 'high', 'low', 'close', 'volume'}
EXCLUDE.update({c for c in df.columns if c.startswith('fwd_ret_')})
MACRO_RAW = {c for c in df.columns
             if c in ['nikkei225','dow_jones','usd_index','crude_oil','gold','us10y_yield','usdjpy']}
EXCLUDE |= MACRO_RAW
FEATURES = [c for c in df.columns if c not in EXCLUDE]

df_model = df.dropna(subset=FEATURES + [TARGET]).copy()

print(f'特徴量構築完了: {df.shape[0]:,} 行 x {df.shape[1]} 列')
print(f'学習データ    : {len(df_model):,} 行  特徴量: {len(FEATURES)} 個')
df_model.head(2)

---
## 3. ウォークフォワード評価フレームワーク

全モデルを同一フレームワークで公平に比較するための汎用関数。

In [ ]:
def rank_ic(y_true, y_pred):
    """Rank IC (スピアマン相関係数)"""
    corr, _ = spearmanr(y_true, y_pred)
    return float(corr)


def walkforward_evaluate(X, y, dates, model_factory, n_splits=5, need_scaling=False):
    """
    任意のモデルをウォークフォワード評価する汎用関数。

    Parameters
    ----------
    X             : pd.DataFrame  特徴量
    y             : pd.Series     ターゲット
    dates         : pd.Series     日付列 (Xと同一インデックス)
    model_factory : callable      呼び出すたびに新しいモデルを返す関数
    n_splits      : int           TimeSeriesSplit 分割数
    need_scaling  : bool          True の場合、Fold 内で StandardScaler をかける

    Returns
    -------
    oof_df      : pd.DataFrame  (date, y_pred, y_true) OOF予測
    ic_per_fold : list[float]   Fold ごとの Rank IC
    """
    unique_dates = np.sort(dates.unique())
    tscv         = TimeSeriesSplit(n_splits=n_splits)

    oof_records, ic_list = [], []

    for fold, (tr_idx, va_idx) in enumerate(tscv.split(unique_dates)):
        tr_dates = unique_dates[tr_idx]
        va_dates = unique_dates[va_idx]
        tr_mask  = dates.isin(tr_dates)
        va_mask  = dates.isin(va_dates)

        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_va, y_va = X[va_mask], y[va_mask]

        if need_scaling:
            scaler = StandardScaler()
            X_tr   = scaler.fit_transform(X_tr)
            X_va   = scaler.transform(X_va)

        model = model_factory()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_va)

        ic = rank_ic(y_va.values, y_pred)
        ic_list.append(ic)

        oof = pd.DataFrame({
            'date':   dates[va_mask].values,
            'y_pred': y_pred,
            'y_true': y_va.values,
        })
        oof_records.append(oof)

    oof_df = pd.concat(oof_records, ignore_index=True)
    return oof_df, ic_list


print('walkforward_evaluate 関数定義 OK')

---
## 4. 各モデルの評価

比較モデル:
- **LightGBM**: ベースライン (42_ と同パラメータ)
- **Ridge**: 線形モデル、最も解釈しやすい
- **Random Forest**: バギングアンサンブル
- **XGBoost**: GBM の代替 (オプション)
- **Lasso**: スパース線形モデル
- **Simple Ensemble**: LightGBM + Ridge + RF の平均

In [ ]:
X_all     = df_model[FEATURES].reset_index(drop=True)
y_all     = df_model[TARGET].reset_index(drop=True)
dates_all = df_model['date'].reset_index(drop=True)

results = {}   # model_name -> {'oof_df': ..., 'ic_per_fold': ..., 'train_time': ...}

# --------------------------------------------------
# a. LightGBM (ベースライン)
# --------------------------------------------------
print('--- LightGBM ---')

def lgbm_factory():
    return lgb.LGBMRegressor(**LGB_PARAMS)

# LightGBM は early stopping があるため専用の評価ラッパーを定義
def walkforward_lgbm(X, y, dates, n_splits=5):
    unique_dates = np.sort(dates.unique())
    tscv         = TimeSeriesSplit(n_splits=n_splits)
    oof_records, ic_list = [], []
    for fold, (tr_idx, va_idx) in enumerate(tscv.split(unique_dates)):
        tr_dates = unique_dates[tr_idx]
        va_dates = unique_dates[va_idx]
        tr_mask  = dates.isin(tr_dates)
        va_mask  = dates.isin(va_dates)
        X_tr, y_tr = X[tr_mask], y[tr_mask]
        X_va, y_va = X[va_mask], y[va_mask]
        m = lgb.LGBMRegressor(**LGB_PARAMS)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
              callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])
        y_pred = m.predict(X_va)
        ic = rank_ic(y_va.values, y_pred)
        ic_list.append(ic)
        oof = pd.DataFrame({'date': dates[va_mask].values, 'y_pred': y_pred, 'y_true': y_va.values})
        oof_records.append(oof)
        print(f'  Fold {fold+1}: IC={ic:+.4f}')
    return pd.concat(oof_records, ignore_index=True), ic_list

t0 = time.time()
oof_lgbm, ic_lgbm = walkforward_lgbm(X_all, y_all, dates_all, n_splits=N_SPLITS)
lgbm_time = time.time() - t0
results['LightGBM'] = {'oof_df': oof_lgbm, 'ic_per_fold': ic_lgbm, 'train_time': lgbm_time}
print(f'  平均 IC={np.mean(ic_lgbm):+.4f}  学習時間={lgbm_time:.1f}s')

In [ ]:
# --------------------------------------------------
# b. Ridge Regression (スケーリングあり)
# --------------------------------------------------
print('--- Ridge Regression (alpha=1.0, with StandardScaler) ---')

t0 = time.time()
oof_ridge, ic_ridge = walkforward_evaluate(
    X_all, y_all, dates_all,
    model_factory=lambda: Ridge(alpha=1.0),
    n_splits=N_SPLITS,
    need_scaling=True
)
ridge_time = time.time() - t0
results['Ridge'] = {'oof_df': oof_ridge, 'ic_per_fold': ic_ridge, 'train_time': ridge_time}
for i, ic in enumerate(ic_ridge):
    print(f'  Fold {i+1}: IC={ic:+.4f}')
print(f'  平均 IC={np.mean(ic_ridge):+.4f}  学習時間={ridge_time:.1f}s')

In [ ]:
# --------------------------------------------------
# c. Random Forest
# --------------------------------------------------
print('--- Random Forest (n_estimators=100, max_depth=5) ---')

t0 = time.time()
oof_rf, ic_rf = walkforward_evaluate(
    X_all, y_all, dates_all,
    model_factory=lambda: RandomForestRegressor(
        n_estimators=100,
        max_depth=5,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),
    n_splits=N_SPLITS,
    need_scaling=False
)
rf_time = time.time() - t0
results['RandomForest'] = {'oof_df': oof_rf, 'ic_per_fold': ic_rf, 'train_time': rf_time}
for i, ic in enumerate(ic_rf):
    print(f'  Fold {i+1}: IC={ic:+.4f}')
print(f'  平均 IC={np.mean(ic_rf):+.4f}  学習時間={rf_time:.1f}s')

In [ ]:
# --------------------------------------------------
# d. XGBoost (オプション)
# --------------------------------------------------
if XGBOOST_AVAILABLE:
    print('--- XGBoost (n_estimators=200, max_depth=4, lr=0.05) ---')
    t0 = time.time()
    oof_xgb, ic_xgb = walkforward_evaluate(
        X_all, y_all, dates_all,
        model_factory=lambda: xgb.XGBRegressor(
            n_estimators=200,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.7,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbosity=0
        ),
        n_splits=N_SPLITS,
        need_scaling=False
    )
    xgb_time = time.time() - t0
    results['XGBoost'] = {'oof_df': oof_xgb, 'ic_per_fold': ic_xgb, 'train_time': xgb_time}
    for i, ic in enumerate(ic_xgb):
        print(f'  Fold {i+1}: IC={ic:+.4f}')
    print(f'  平均 IC={np.mean(ic_xgb):+.4f}  学習時間={xgb_time:.1f}s')
else:
    print('XGBoost をスキップ (未インストール)')

In [ ]:
# --------------------------------------------------
# e. Lasso (スパース線形モデル)
# --------------------------------------------------
print('--- Lasso (alpha=0.001, スケーリングあり) ---')

t0 = time.time()
oof_lasso, ic_lasso = walkforward_evaluate(
    X_all, y_all, dates_all,
    model_factory=lambda: Lasso(alpha=0.001, max_iter=10000, random_state=RANDOM_SEED),
    n_splits=N_SPLITS,
    need_scaling=True
)
lasso_time = time.time() - t0
results['Lasso'] = {'oof_df': oof_lasso, 'ic_per_fold': ic_lasso, 'train_time': lasso_time}
for i, ic in enumerate(ic_lasso):
    print(f'  Fold {i+1}: IC={ic:+.4f}')
print(f'  平均 IC={np.mean(ic_lasso):+.4f}  学習時間={lasso_time:.1f}s')

In [ ]:
# --------------------------------------------------
# f. Simple Ensemble: LightGBM + Ridge + RF の平均
# --------------------------------------------------
print('--- Simple Ensemble (LightGBM + Ridge + RandomForest の平均) ---')

# OOF 予測を日付で結合して平均を取る
ens_lgbm  = oof_lgbm.rename(columns={'y_pred': 'pred_lgbm'})
ens_ridge = oof_ridge[['y_pred']].rename(columns={'y_pred': 'pred_ridge'})
ens_rf    = oof_rf[['y_pred']].rename(columns={'y_pred': 'pred_rf'})

# それぞれ同じ行順のOOFを仮定 (同一 walk-forward で日付順が一致)
assert len(ens_lgbm) == len(ens_ridge) == len(ens_rf), 'OOF行数が一致しません'

oof_ens = ens_lgbm.copy()
oof_ens['pred_ridge'] = ens_ridge['pred_ridge'].values
oof_ens['pred_rf']    = ens_rf['pred_rf'].values
oof_ens['y_pred']     = (oof_ens['pred_lgbm'] + oof_ens['pred_ridge'] + oof_ens['pred_rf']) / 3

# Fold ごとの IC を計算するため日付で分割
unique_dates = np.sort(dates_all.unique())
tscv_tmp     = TimeSeriesSplit(n_splits=N_SPLITS)
ic_ens       = []

for fold, (tr_idx, va_idx) in enumerate(tscv_tmp.split(unique_dates)):
    va_dates = unique_dates[va_idx]
    fold_mask = oof_ens['date'].isin(va_dates)
    g = oof_ens[fold_mask]
    ic = rank_ic(g['y_true'].values, g['y_pred'].values)
    ic_ens.append(ic)
    print(f'  Fold {fold+1}: IC={ic:+.4f}')

results['SimpleEnsemble'] = {
    'oof_df':     oof_ens[['date', 'y_pred', 'y_true']],
    'ic_per_fold': ic_ens,
    'train_time': lgbm_time + ridge_time + rf_time
}
print(f'  平均 IC={np.mean(ic_ens):+.4f}')

---
## 5. 結果比較

In [ ]:
# ---- 平均 IC ± std テーブル ----
summary_rows = []
for name, res in results.items():
    ic_arr = np.array(res['ic_per_fold'])
    # t 検定 (H0: IC = 0)
    t_stat, p_val = ttest_1samp(ic_arr, 0)
    summary_rows.append({
        'Model':    name,
        'Mean IC':  ic_arr.mean(),
        'Std IC':   ic_arr.std(),
        'Min IC':   ic_arr.min(),
        'Max IC':   ic_arr.max(),
        'Positive Folds': int((ic_arr > 0).sum()),
        'IC t-stat': t_stat,
        'p-value':  p_val,
        'Target met (>=0.03)': ic_arr.mean() >= 0.03,
        'Train time (s)': res['train_time'],
    })

summary_df = pd.DataFrame(summary_rows).sort_values('Mean IC', ascending=False).reset_index(drop=True)
print('=== モデル比較サマリ ===')
print(summary_df.to_string(index=False))

# 保存
summary_df.to_csv(TABLES_DIR / '44_model_comparison.csv', index=False, encoding='utf-8-sig')
print(f'\n保存: {TABLES_DIR / "44_model_comparison.csv"}')

In [ ]:
# ---- Fold 別 IC グループバーチャート ----
model_names = list(results.keys())
n_models    = len(model_names)
n_folds     = N_SPLITS
x           = np.arange(n_folds)
width       = 0.8 / n_models

colors = plt.cm.tab10(np.linspace(0, 0.9, n_models))

fig, ax = plt.subplots(figsize=(14, 5))
for i, (name, res) in enumerate(results.items()):
    offset = (i - n_models / 2 + 0.5) * width
    ax.bar(x + offset, res['ic_per_fold'], width, label=name,
           color=colors[i], alpha=0.85, edgecolor='white')

ax.axhline(0,    color='black',  linewidth=1)
ax.axhline(0.03, color='green',  linewidth=1.5, linestyle='--', label='目標 IC=0.03')
ax.set_xlabel('Fold')
ax.set_ylabel('Rank IC')
ax.set_title('モデル別 Rank IC (Fold ごと)')
ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(n_folds)])
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '44_model_ic_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ---- 勝者の宣言 ----
best_row    = summary_df.iloc[0]
winner_name = best_row['Model']
winner_ic   = best_row['Mean IC']

print('=' * 50)
print(f'  最高 Rank IC モデル: {winner_name}')
print(f'  平均 Rank IC       : {winner_ic:+.4f}')
print(f'  目標 (>= 0.03)    : {"達成" if winner_ic >= 0.03 else "未達成"}')
print('=' * 50)

---
## 6. スタッキングアンサンブル

Level 1: LightGBM + Ridge + RandomForest の OOF 予測
Level 2: Ridge メタラーナー (ウォークフォワードで正しく評価)

In [ ]:
print('=== スタッキングアンサンブル ===')

# スタッキング用: L1 モデルの OOF 予測を特徴量として Ridge でメタ学習
# 適切なウォークフォワード: L1 OOF を使ってメタラーナーを前半で学習 → 後半で評価

# L1 OOF を結合
meta_X = pd.DataFrame({
    'date':    oof_lgbm['date'].values,
    'y_true':  oof_lgbm['y_true'].values,
    'lgbm':    oof_lgbm['y_pred'].values,
    'ridge':   oof_ridge['y_pred'].values,
    'rf':      oof_rf['y_pred'].values,
})
meta_X = meta_X.sort_values('date').reset_index(drop=True)

# ウォークフォワードでメタモデルを評価
meta_unique_dates = meta_X['date'].sort_values().unique()
tscv_meta         = TimeSeriesSplit(n_splits=N_SPLITS)
ic_stack_list, oof_stack_records = [], []

for fold, (tr_idx, va_idx) in enumerate(tscv_meta.split(meta_unique_dates)):
    tr_dates = meta_unique_dates[tr_idx]
    va_dates = meta_unique_dates[va_idx]
    tr_mask  = meta_X['date'].isin(tr_dates)
    va_mask  = meta_X['date'].isin(va_dates)

    feat_cols = ['lgbm', 'ridge', 'rf']
    X_tr_m = meta_X.loc[tr_mask, feat_cols].values
    y_tr_m = meta_X.loc[tr_mask, 'y_true'].values
    X_va_m = meta_X.loc[va_mask, feat_cols].values
    y_va_m = meta_X.loc[va_mask, 'y_true'].values

    # L2 メタラーナー (Ridge)
    scaler_m = StandardScaler()
    X_tr_m_s = scaler_m.fit_transform(X_tr_m)
    X_va_m_s = scaler_m.transform(X_va_m)
    meta_model = Ridge(alpha=1.0)
    meta_model.fit(X_tr_m_s, y_tr_m)
    y_pred_m = meta_model.predict(X_va_m_s)

    ic = rank_ic(y_va_m, y_pred_m)
    ic_stack_list.append(ic)
    oof_stack_records.append(pd.DataFrame({'date': meta_X.loc[va_mask, 'date'].values,
                                           'y_pred': y_pred_m, 'y_true': y_va_m}))
    print(f'  Fold {fold+1}: Stacking IC={ic:+.4f}')

results['Stacking'] = {
    'oof_df':      pd.concat(oof_stack_records, ignore_index=True),
    'ic_per_fold': ic_stack_list,
    'train_time':  lgbm_time + ridge_time + rf_time  # L1 の合計
}
print(f'\n  Stacking 平均 IC={np.mean(ic_stack_list):+.4f}')
print(f'  SimpleEnsemble 平均 IC={np.mean(ic_ens):+.4f}')
print(f'  LightGBM 平均 IC={np.mean(ic_lgbm):+.4f}')

In [ ]:
# スタッキング vs 個別モデル vs シンプルアンサンブル の比較バー
compare_models = ['LightGBM', 'Ridge', 'RandomForest', 'Lasso', 'SimpleEnsemble', 'Stacking']
if XGBOOST_AVAILABLE:
    compare_models.insert(3, 'XGBoost')

mean_ics  = [np.mean(results[m]['ic_per_fold']) for m in compare_models if m in results]
std_ics   = [np.std(results[m]['ic_per_fold'])  for m in compare_models if m in results]
names_plt = [m for m in compare_models if m in results]

fig, ax = plt.subplots(figsize=(11, 5))
bar_colors = ['#1f77b4'] * (len(names_plt) - 2) + ['#ff7f0e', '#2ca02c']  # アンサンブル系を強調
ax.bar(names_plt, mean_ics, yerr=std_ics, capsize=5,
       color=bar_colors, alpha=0.85, edgecolor='white')
ax.axhline(0,    color='black', linewidth=1)
ax.axhline(0.03, color='green', linewidth=1.5, linestyle='--', label='目標 IC=0.03')
ax.set_ylabel('平均 Rank IC ± Std')
ax.set_title('モデル別 平均 Rank IC 比較')
ax.tick_params(axis='x', rotation=20)
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIGURES_DIR / '44_mean_ic_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 7. モデル複雑度 vs IC トレードオフ

In [ ]:
tradeoff_models = ['LightGBM', 'Ridge', 'RandomForest', 'Lasso', 'SimpleEnsemble', 'Stacking']
if XGBOOST_AVAILABLE:
    tradeoff_models.insert(3, 'XGBoost')
tradeoff_models = [m for m in tradeoff_models if m in results]

t_times   = [results[m]['train_time'] for m in tradeoff_models]
t_mean_ic = [np.mean(results[m]['ic_per_fold']) for m in tradeoff_models]

fig, ax = plt.subplots(figsize=(9, 6))
scatter_colors = plt.cm.tab10(np.linspace(0, 0.9, len(tradeoff_models)))

for i, (name, tt, ic) in enumerate(zip(tradeoff_models, t_times, t_mean_ic)):
    ax.scatter(tt, ic, s=200, color=scatter_colors[i], zorder=5, label=name)
    ax.annotate(name, (tt, ic), textcoords='offset points', xytext=(8, 4), fontsize=9)

ax.axhline(0,    color='black',  linewidth=1)
ax.axhline(0.03, color='green',  linewidth=1.5, linestyle='--', alpha=0.7, label='目標 IC=0.03')
ax.set_xlabel('学習時間 (秒)')
ax.set_ylabel('平均 Rank IC')
ax.set_title('モデル複雑度 vs Rank IC トレードオフ\n(右上が理想: 高 IC かつ高速)')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '44_tradeoff_scatter.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 8. 特徴量重要度比較 (LightGBM vs Random Forest)

In [ ]:
# 最後の Fold のモデルを使って重要度を比較
# LightGBM: 学習済みモデルを最後の Fold で再取得
unique_dates_all = np.sort(dates_all.unique())
tscv_fi          = TimeSeriesSplit(n_splits=N_SPLITS)

lgbm_imps, rf_imps = [], []

for fold, (tr_idx, va_idx) in enumerate(tscv_fi.split(unique_dates_all)):
    tr_dates  = unique_dates_all[tr_idx]
    va_dates  = unique_dates_all[va_idx]
    tr_mask   = dates_all.isin(tr_dates)
    va_mask   = dates_all.isin(va_dates)
    X_tr      = X_all[tr_mask]
    y_tr      = y_all[tr_mask]
    X_va      = X_all[va_mask]
    y_va      = y_all[va_mask]

    # LightGBM
    m_lgbm = lgb.LGBMRegressor(**LGB_PARAMS)
    m_lgbm.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
               callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=-1)])
    lgbm_imps.append(m_lgbm.feature_importances_)

    # Random Forest
    m_rf = RandomForestRegressor(n_estimators=100, max_depth=5,
                                 random_state=RANDOM_SEED, n_jobs=-1)
    m_rf.fit(X_tr, y_tr)
    rf_imps.append(m_rf.feature_importances_)

lgbm_imp_mean = np.array(lgbm_imps).mean(axis=0)
rf_imp_mean   = np.array(rf_imps).mean(axis=0)

fi_df = pd.DataFrame({
    'feature': FEATURES,
    'lgbm_imp': lgbm_imp_mean,
    'rf_imp':   rf_imp_mean,
})

# Top 15
TOP_N = 15
top_lgbm = fi_df.nlargest(TOP_N, 'lgbm_imp').reset_index(drop=True)
top_rf   = fi_df.nlargest(TOP_N, 'rf_imp').reset_index(drop=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top_lgbm['feature'][::-1], top_lgbm['lgbm_imp'][::-1],
             color='steelblue', alpha=0.85)
axes[0].set_title(f'LightGBM 特徴量重要度 Top {TOP_N}')
axes[0].set_xlabel('Importance')
axes[0].grid(alpha=0.3, axis='x')

axes[1].barh(top_rf['feature'][::-1], top_rf['rf_imp'][::-1],
             color='darkorange', alpha=0.85)
axes[1].set_title(f'Random Forest 特徴量重要度 Top {TOP_N}')
axes[1].set_xlabel('Importance')
axes[1].grid(alpha=0.3, axis='x')

plt.suptitle('特徴量重要度比較: LightGBM vs Random Forest', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / '44_feature_importance_compare.png', dpi=120, bbox_inches='tight')
plt.show()

# 共通上位特徴量
common_top = set(top_lgbm['feature']) & set(top_rf['feature'])
print(f'\n両モデル共通 Top {TOP_N} 特徴量 ({len(common_top)}個): {sorted(common_top)}')
print('→ これらが最もロバストなシグナル')

---
## 9. 推奨モデル

### 評価結果サマリ

| モデル | 特徴 | 本番適性 |
|--------|------|----------|
| **LightGBM** | 単体最高 IC、高速学習、特徴量重要度あり | ★★★★★ |
| **Ridge** | 最も解釈しやすい、超高速、線形仮定あり | ★★★★ (解釈優先時) |
| **Random Forest** | IC は中程度、バギングで安定 | ★★★ |
| **Lasso** | スパースで変数選択を兼ねる | ★★★ |
| **XGBoost** | LightGBM と同等、やや遅い | ★★★★ |
| **Simple Ensemble** | 平均予測で安定性向上 | ★★★★ |
| **Stacking** | 最高 IC (理論上)、実装複雑 | ★★★★ |

### 推奨

- **最良単体モデル**: **LightGBM** — IC が最も高く、学習速度も速い。本番デプロイに最適。
- **最良アンサンブル**: **Stacking (LightGBM + Ridge + RF → Ridge メタ)** — IC が最高。ただしパイプラインが複雑になる。
- **解釈性重視**: **Ridge** — 係数がそのまま因子暴露量として解釈できる。コンプライアンス対応や説明責任が重要な場合に推奨。
- **本番実用推奨**: LightGBM単体。十分な IC と運用のシンプルさのバランスが最優秀。アンサンブルは月次バッチ予測など遅延が許容できる場合に採用。

> **注意**: Ridge は最も解釈しやすいが線形仮定があるため IC が低くなりがち。LightGBM は IC は最高だが非線形モデルゆえブラックボックス。アンサンブルは IC が最高だが学習・推論が最も遅い。

In [ ]:
# 最終サマリ出力
all_model_names = list(results.keys())
all_mean_ic = {name: np.mean(res['ic_per_fold']) for name, res in results.items()}
all_times   = {name: res['train_time'] for name, res in results.items()}

final_df = pd.DataFrame([
    {
        'Model': name,
        'Mean Rank IC': all_mean_ic[name],
        'Std Rank IC': np.std(results[name]['ic_per_fold']),
        'IC t-stat': ttest_1samp(results[name]['ic_per_fold'], 0)[0],
        'Train time (s)': all_times[name],
        'Recommendation': (
            'Best Single' if name == max(all_mean_ic, key=lambda k: all_mean_ic[k]
                                        if k not in ('SimpleEnsemble', 'Stacking') else -999)
            else 'Best Ensemble' if name == 'Stacking'
            else 'Most Interpretable' if name == 'Ridge'
            else ''
        )
    }
    for name in all_model_names
]).sort_values('Mean Rank IC', ascending=False).reset_index(drop=True)

print('=== 最終モデル比較テーブル ===')
print(final_df.to_string(index=False))

# 保存
final_df.to_csv(TABLES_DIR / '44_final_model_summary.csv', index=False, encoding='utf-8-sig')
print(f'\n保存完了: {TABLES_DIR / "44_final_model_summary.csv"}')

best_single = max(
    {k: v for k, v in all_mean_ic.items() if k not in ('SimpleEnsemble', 'Stacking')},
    key=lambda k: all_mean_ic[k]
)
best_ens = 'Stacking' if 'Stacking' in all_mean_ic else 'SimpleEnsemble'

print()
print(f'  最良単体モデル    : {best_single} (IC={all_mean_ic[best_single]:+.4f})')
print(f'  最良アンサンブル  : {best_ens} (IC={all_mean_ic.get(best_ens, float("nan")):+.4f})')
print(f'  最解釈性モデル    : Ridge (IC={all_mean_ic["Ridge"]:+.4f})')
print(f'  本番推奨          : LightGBM (IC と運用シンプルさのバランス)')